# Lesson 17: Open-Source & Open-Weight Models on Hugging Face

In this notebook, you will:
1. Load an open-weight model from **Hugging Face `transformers`** (`Qwen/Qwen2.5-0.5B-Instruct` or `google/gemma-2-2b-it`).
2. Inspect its attention configuration (`num_attention_heads` vs `num_key_value_heads` GQA ratio).
3. Benchmark inference latency and tokens/second on Google Colab.

In [ ]:
!pip install -q -U transformers accelerate torch

In [ ]:
import time, torch
from transformers import AutoConfig, AutoModelForCausalLM, AutoTokenizer

model_id = "Qwen/Qwen2.5-0.5B-Instruct"
cfg = AutoConfig.from_pretrained(model_id)
print(f"Model: {model_id} | Q heads: {cfg.num_attention_heads} | KV heads (GQA): {cfg.num_key_value_heads}")

tokenizer = AutoTokenizer.from_pretrained(model_id)
device = "cuda" if torch.cuda.is_available() else "cpu"
model = AutoModelForCausalLM.from_pretrained(model_id, torch_dtype="auto").to(device)

inputs = tokenizer("Explain Multi-Query Attention in two sentences:", return_tensors="pt").to(device)
t0 = time.perf_counter()
out = model.generate(**inputs, max_new_tokens=60)
dt = time.perf_counter() - t0
print(tokenizer.decode(out[0], skip_special_tokens=True))
print(f"Elapsed: {dt:.2f}s on {device}")